# QLoRA Fine-Tuning (Colab)

Fine-tunes `mistralai/Mistral-7B-Instruct-v0.3` on the tri-modal scenario dataset using the **fixed** pipeline (`fine_tune.py`). No training logic lives in this notebook — it ships the repo modules and runs the script, so the prompt/precision/masking fixes always apply.

Steps: GPU check -> install -> ship code -> upload `training_data.jsonl` -> train -> download the adapter + `test_inputs.jsonl`.

**Precision is auto-selected:** bf16 on A100, fp16 on T4.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('bf16 supported (A100=True, T4=False):', torch.cuda.is_bf16_supported())
else:
    print('No GPU. Runtime -> Change runtime type -> T4 GPU')

In [ ]:
!pip install -q -r requirements_finetune.txt
print('Installed fine-tuning requirements.')

In [ ]:
# Ship the fixed repo modules into this runtime (source of truth).
# These are the repaired pipeline files.
import pathlib
pathlib.Path("prompt_format.py").write_text("\"\"\"\nprompt_format.py\n----------------\nTHE single source of truth for the prompt that wraps each training record and\neach inference request. Training (finetuning.py) and every inference path\n(run_inference.py, evaluation/evaluate.py, the generated notebooks) all import\n``build_chat`` from here, so the train-time and infer-time prompts can never\ndrift apart. Prompt drift was the original root cause of the off-format model\noutput, so this module is deliberately small and shared.\n\nWhat the prompt looks like for Mistral-7B-Instruct-v0.3\n------------------------------------------------------\nMistral's chat template has no separate ``system`` role that survives a single\nuser turn, so we fold the full ``instruction`` (the tri-modal formatting rules)\ninto the user turn together with the record's ``input``::\n\n    <s>[INST] {instruction}\\\\n\\\\n{input}[/INST]                 # inference prompt\n    <s>[INST] {instruction}\\\\n\\\\n{input}[/INST] {output}</s>    # full training text\n\nThe inference prompt is, by construction, a strict prefix of the training text,\nwhich is exactly what the model needs to continue the completion at inference.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Optional\n\n\n# Mistral-7B-Instruct-v0.3 special tokens. Used only by the no-dependency manual\n# fallback below; when a real tokenizer is supplied we read these from it.\nDEFAULT_BOS = \"<s>\"\nDEFAULT_EOS = \"</s>\"\n\n# The marker that separates the prompt from the completion. Loss is computed only\n# on the tokens that follow this marker (see finetuning.py completion-only setup).\nRESPONSE_MARKER = \"[/INST]\"\n\n\ndef build_messages(record: dict, for_training: bool) -> list[dict]:\n    \"\"\"\n    Build the chat ``messages`` list for one record.\n\n    The full ``instruction`` field (all the [NATURAL LANGUAGE] / [MATHEMATICAL\n    EXPRESSION] / [ATTACK GRAPH] rules) is folded into the user turn — never the\n    short one-liner that older code used.\n    \"\"\"\n    instruction = (record.get(\"instruction\") or \"\").rstrip()\n    input_text = record.get(\"input\") or \"\"\n    user_content = f\"{instruction}\\n\\n{input_text}\"\n\n    messages = [{\"role\": \"user\", \"content\": user_content}]\n    if for_training:\n        output = (record.get(\"output\") or \"\").strip()\n        messages.append({\"role\": \"assistant\", \"content\": output})\n    return messages\n\n\ndef _manual_chat(record: dict, for_training: bool,\n                 bos: str = DEFAULT_BOS, eos: str = DEFAULT_EOS) -> str:\n    \"\"\"\n    Dependency-free fallback that reproduces Mistral's chat template byte-for-byte.\n\n    Mirrors output/chat_template.jinja:\n      user turn      -> \"[INST] \" + content + \"[/INST]\"   (bos prepended once)\n      assistant turn -> \" \" + content.strip() + eos\n    Used when no tokenizer is available (e.g. ``--check_data`` on a machine with\n    no transformers installed). The infer string stays a prefix of the train\n    string here too.\n    \"\"\"\n    instruction = (record.get(\"instruction\") or \"\").rstrip()\n    input_text = record.get(\"input\") or \"\"\n    prompt = f\"{bos}[INST] {instruction}\\n\\n{input_text}[/INST]\"\n    if not for_training:\n        return prompt\n    output = (record.get(\"output\") or \"\").strip()\n    return f\"{prompt} {output}{eos}\"\n\n\ndef build_chat(record: dict, tokenizer: Optional[object], for_training: bool) -> str:\n    \"\"\"\n    Return the canonical prompt string for ``record``.\n\n    Parameters\n    ----------\n    record       : dict with at least ``instruction`` and ``input`` (and\n                   ``output`` when ``for_training`` is True).\n    tokenizer    : a HF tokenizer (uses ``apply_chat_template``) or ``None``\n                   (falls back to the manual template above).\n    for_training : True  -> include the assistant turn + EOS (full training text)\n                   False -> stop after ``[/INST]`` (the generation prompt)\n\n    Guarantee: ``build_chat(rec, tok, False)`` is always a strict prefix of\n    ``build_chat(rec, tok, True)`` for the same record/tokenizer.\n    \"\"\"\n    if tokenizer is not None and hasattr(tokenizer, \"apply_chat_template\"):\n        try:\n            return tokenizer.apply_chat_template(\n                build_messages(record, for_training),\n                tokenize=False,\n                add_generation_prompt=not for_training,\n            )\n        except Exception:\n            # Some odd template/version combo — fall back to the manual template,\n            # using the tokenizer's real special tokens so it stays consistent.\n            bos = getattr(tokenizer, \"bos_token\", None) or DEFAULT_BOS\n            eos = getattr(tokenizer, \"eos_token\", None) or DEFAULT_EOS\n            return _manual_chat(record, for_training, bos, eos)\n    return _manual_chat(record, for_training)\n\n\ndef assert_parity(records: list[dict], tokenizer: Optional[object] = None) -> None:\n    \"\"\"\n    Assert that the inference prompt is a strict prefix of the full training text\n    for every record. This is the train/inference-parity guard (fix A acceptance).\n    Raises AssertionError on the first violation.\n    \"\"\"\n    for i, rec in enumerate(records):\n        infer = build_chat(rec, tokenizer, for_training=False)\n        train = build_chat(rec, tokenizer, for_training=True)\n        assert train.startswith(infer), (\n            f\"Parity violated for record {i} \"\n            f\"(source_id={rec.get('_meta', {}).get('source_id')}): \"\n            \"inference prompt is not a prefix of the training text.\"\n        )\n        assert len(train) > len(infer), (\n            f\"Record {i} produced an empty completion span.\"\n        )\n\n\ndef response_template_ids(tokenizer, sample_text: str, marker: str = RESPONSE_MARKER) -> list[int]:\n    \"\"\"\n    Compute the token ids of ``marker`` (``[/INST]``) **in context**.\n\n    The same string can tokenize differently standalone vs. mid-sequence, which\n    is the classic footgun that makes DataCollatorForCompletionOnlyLM silently\n    mask the entire example. We therefore diff the tokenization of the text up to\n    *and including* the marker against the text up to *just before* it, and use\n    the trailing ids — the ids the marker actually contributes in this context.\n\n    Requires a real tokenizer; only called on the training/smoke path.\n    \"\"\"\n    cut = sample_text.index(marker)\n    end = cut + len(marker)\n    before_ids = tokenizer(sample_text[:cut], add_special_tokens=False)[\"input_ids\"]\n    upto_ids = tokenizer(sample_text[:end], add_special_tokens=False)[\"input_ids\"]\n\n    # The marker's in-context ids are whatever was appended after `before_ids`.\n    # Guard against rare boundary re-tokenization of the last shared token.\n    k = 0\n    while k < len(before_ids) and k < len(upto_ids) and before_ids[k] == upto_ids[k]:\n        k += 1\n    ids = upto_ids[k:]\n    if not ids:\n        # Fallback: standalone encoding (better than an empty template).\n        ids = tokenizer(marker, add_special_tokens=False)[\"input_ids\"]\n    return ids\n", encoding='utf-8')
pathlib.Path("data_split.py").write_text("\"\"\"\ndata_split.py\n-------------\nTHE single source of truth for splitting the dataset into train / validation /\ntest. Persisting the split here (and reading it everywhere) removes the old\ntrain/eval mismatch where finetuning.py did its own random split while\nmake_test_inputs.py treated a different file's tail as \"unseen\" — so the\n\"unseen\" test set was not guaranteed unseen.\n\nHow it works\n------------\n* The split is computed once, deterministically, from a single seed, keyed on\n  each record's stable ``_meta.source_id``.\n* The held-out **test** records are written to ``evaluation/test_inputs.jsonl``.\n* finetuning.py trains on the **complement** of that file (train + val only);\n  make_test_inputs.py creates it; evaluate.py reads it. Everyone shares one\n  artifact, so train ∩ test is provably empty.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport random\nfrom pathlib import Path\nfrom typing import Optional\n\n\nDEFAULT_TEST_INPUTS = \"evaluation/test_inputs.jsonl\"\n\n\ndef load_records(jsonl_path: str | Path) -> list[dict]:\n    \"\"\"Read a JSONL dataset, skipping blank / malformed lines.\"\"\"\n    records = []\n    with open(jsonl_path, \"r\", encoding=\"utf-8\") as f:\n        for line in f:\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                records.append(json.loads(line))\n            except json.JSONDecodeError:\n                continue\n    return records\n\n\ndef source_id(record: dict, idx: int) -> str:\n    \"\"\"Stable id for a record: its _meta.source_id, else a positional fallback.\"\"\"\n    sid = (record.get(\"_meta\") or {}).get(\"source_id\")\n    return sid if sid else f\"idx-{idx}\"\n\n\ndef _annotate(records: list[dict]) -> list[dict]:\n    \"\"\"Attach the original index + canonical source_id to each record (copy).\"\"\"\n    out = []\n    for i, rec in enumerate(records):\n        item = dict(rec)\n        item[\"idx\"] = i\n        item[\"source_id\"] = source_id(rec, i)\n        out.append(item)\n    return out\n\n\ndef compute_split(records: list[dict], test_size: int = 10,\n                  val_frac: float = 0.15, seed: int = 42) -> dict:\n    \"\"\"\n    Deterministically split annotated records into test / val / train.\n\n    test_size : number of held-out test records (default 10).\n    val_frac  : fraction of the *remaining* pool used for validation\n                (always at least 1 record so eval_loss can be computed).\n    Returns {\"train\": [...], \"val\": [...], \"test\": [...]} with records sorted by\n    original index within each split.\n    \"\"\"\n    annotated = _annotate(records)\n    n = len(annotated)\n    if test_size >= n:\n        raise ValueError(f\"test_size={test_size} >= dataset size {n}\")\n\n    order = list(range(n))\n    random.Random(seed).shuffle(order)\n\n    test_idx = order[:test_size]\n    pool_idx = order[test_size:]\n    val_count = max(1, round(val_frac * len(pool_idx)))\n    val_count = min(val_count, len(pool_idx) - 1)  # keep at least 1 for train\n    val_idx = pool_idx[:val_count]\n    train_idx = pool_idx[val_count:]\n\n    by_idx = {item[\"idx\"]: item for item in annotated}\n    pick = lambda idxs: [by_idx[i] for i in sorted(idxs)]\n    return {\"train\": pick(train_idx), \"val\": pick(val_idx), \"test\": pick(test_idx)}\n\n\n# ── Persisted test-set artifact ────────────────────────────────────────────────\n\n# Fields we keep in test_inputs.jsonl. `output` is the reference completion (for\n# eval comparison only — never fed to the model as input).\nTEST_FIELDS = (\"idx\", \"source_id\", \"instruction\", \"input\", \"output\")\n\n\ndef write_test_inputs(path: str | Path, test_records: list[dict]) -> None:\n    \"\"\"Persist the held-out test records to a JSONL artifact.\"\"\"\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"w\", encoding=\"utf-8\") as f:\n        for rec in test_records:\n            row = {k: rec.get(k, \"\") for k in TEST_FIELDS}\n            f.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n\n\ndef read_test_inputs(path: str | Path) -> list[dict]:\n    \"\"\"Read the held-out test artifact.\"\"\"\n    return load_records(path)\n\n\ndef ids_of(records: list[dict]) -> set:\n    \"\"\"Set of canonical ids (idx, source_id) tuples for intersection checks.\"\"\"\n    out = set()\n    for i, rec in enumerate(records):\n        out.add((rec.get(\"idx\", i), rec.get(\"source_id\") or source_id(rec, i)))\n    return out\n\n\ndef ensure_test_inputs(data_path: str | Path,\n                       test_inputs_path: str | Path = DEFAULT_TEST_INPUTS,\n                       test_size: int = 10, val_frac: float = 0.15,\n                       seed: int = 42) -> dict:\n    \"\"\"\n    Return the split for ``data_path``, creating ``test_inputs_path`` from the\n    deterministic split if it does not exist yet. When it already exists, the\n    test split is taken from the persisted artifact (source of truth) and the\n    train/val pool is the complement, so all callers agree.\n    \"\"\"\n    records = load_records(data_path)\n    split = compute_split(records, test_size=test_size, val_frac=val_frac, seed=seed)\n\n    test_path = Path(test_inputs_path)\n    if not test_path.exists():\n        write_test_inputs(test_path, split[\"test\"])\n        return split\n\n    # Artifact exists. If it matches the seed-derived split exactly, use the\n    # canonical split so train/val are identical regardless of who created the\n    # file first (finetuning vs make_test_inputs).\n    persisted_ids = {row.get(\"source_id\") for row in read_test_inputs(test_path)}\n    if persisted_ids == {r[\"source_id\"] for r in split[\"test\"]}:\n        return split\n\n    # A custom / hand-edited artifact: respect it and derive the complement.\n    print(f\"[i] {test_path} differs from the seed-derived split; using the persisted test set.\")\n    annotated = _annotate(records)\n    pool = [r for r in annotated if r[\"source_id\"] not in persisted_ids]\n    test = [r for r in annotated if r[\"source_id\"] in persisted_ids]\n    order = list(range(len(pool)))\n    random.Random(seed).shuffle(order)\n    val_count = max(1, min(round(val_frac * len(pool)), len(pool) - 1))\n    val_idx = set(order[:val_count])\n    val = [pool[i] for i in sorted(val_idx)]\n    train = [pool[i] for i in range(len(pool)) if i not in val_idx]\n    return {\"train\": train, \"val\": val, \"test\": test}\n", encoding='utf-8')
pathlib.Path("finetuning.py").write_text("\"\"\"\nfinetuning.py\n-------------\nQLoRA fine-tuning of Mistral-7B-Instruct on the tri-modal cyber attack\nscenario dataset generated by generate_outputs.py.\n\nWhat is QLoRA?\n  Q  = Quantization: loads the model in 4-bit precision (uses ~4-5GB VRAM\n       instead of ~28GB), making it runnable on a free Colab/Kaggle GPU.\n  Lo = Low-Rank: instead of updating all 7 billion weights, we attach\n       small \"adapter\" matrices (LoRA) to key layers and only train those.\n  RA = Rank Adaptation: the adapters have rank r (default 16), meaning each\n       adapter is a pair of tiny matrices (r x hidden_dim) — very few params.\n\nResult: a fine-tuned model that knows the cybersecurity scenario format\nwithout needing a $10,000 GPU.\n\nUsage (run from scenario_generation/):\n    python fine_tuning/fine_tune.py \\\n        --data  dataset/training_data.jsonl \\\n        --out   fine_tuning/output/\n\nOffline probes (no GPU / no ML stack required):\n    python fine_tuning/fine_tune.py --check_data        # data + prompt parity\n    python fine_tuning/fine_tune.py --smoke --out /tmp/s # 3-step CUDA-guarded run\n\nRequirements:\n    pip install -r fine_tuning/requirements_finetune.txt\n\nNotes on portability:\n  * The heavy ML imports (torch, transformers, trl, peft, datasets) are done\n    LAZILY inside the functions that need them, so --help, --check_data and the\n    GPU-less --smoke path work on a machine with no ML stack installed.\n  * TRL/transformers APIs differ across versions; we DETECT the available\n    field/argument names at runtime and branch, rather than hardcoding.\n\"\"\"\n\nimport argparse\nimport math\nimport sys\nfrom pathlib import Path\n\n# Local shared modules (same directory). These are dependency-free so they import\n# fine even without the ML stack.\nimport prompt_format\nimport data_split\n\n\n# ── Constants ─────────────────────────────────────────────────────────────────\n\nBASE_MODEL   = \"mistralai/Mistral-7B-Instruct-v0.3\"\nMAX_SEQ_LEN  = 2048   # max token length per training example (see fix C)\n\n# LoRA target-module presets.\nALL_LINEAR = [\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\", \"gate_proj\", \"up_proj\", \"down_proj\"]\nATTN_ONLY  = [\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\"]\n\nDEFAULT_TEST_INPUTS = \"evaluation/test_inputs.jsonl\"\n\n# The three section headers the model must emit, in order (used by the self-test).\nREQUIRED_HEADERS = [\"[NATURAL LANGUAGE]\", \"[MATHEMATICAL EXPRESSION]\", \"[ATTACK GRAPH]\"]\n\n\n# ── Dependency check (only invoked on the real-training / GPU paths) ───────────\n\ndef check_imports():\n    \"\"\"Friendly check for the heavy ML stack; called only when we actually train.\"\"\"\n    missing = []\n    for pkg, import_name in [\n        (\"transformers\", \"transformers\"),\n        (\"peft\",         \"peft\"),\n        (\"trl\",          \"trl\"),\n        (\"bitsandbytes\", \"bitsandbytes\"),\n        (\"datasets\",     \"datasets\"),\n        (\"accelerate\",   \"accelerate\"),\n        (\"torch\",        \"torch\"),\n    ]:\n        try:\n            __import__(import_name)\n        except ImportError:\n            missing.append(pkg)\n    if missing:\n        print(\"[!] Missing packages. Run:\")\n        print(f\"    pip install {' '.join(missing)}\")\n        sys.exit(1)\n\n\ndef estimate_tokens(text: str) -> int:\n    \"\"\"Rough token-count estimate (~4 chars/token) used only when no tokenizer.\"\"\"\n    return max(1, round(len(text) / 4))\n\n\n# ── Dataset loading (shared split + canonical prompt) ─────────────────────────\n\ndef load_dataset_split(data_path, tokenizer, test_inputs_path, test_size, val_frac, seed):\n    \"\"\"\n    Build the train/validation HuggingFace Datasets.\n\n    The held-out TEST set is excluded entirely (it lives in test_inputs.jsonl and\n    is used only by evaluation) — we train on its complement. Each example's\n    'text' field is the canonical full training text from prompt_format.build_chat\n    (instruction + input + output, EOS-terminated), identical to what inference\n    will prefix-match.\n    \"\"\"\n    from datasets import Dataset\n\n    split = data_split.ensure_test_inputs(\n        data_path, test_inputs_path,\n        test_size=test_size, val_frac=val_frac, seed=seed,\n    )\n\n    def to_ds(recs):\n        rows = [{\"text\": prompt_format.build_chat(r, tokenizer, for_training=True)}\n                for r in recs]\n        return Dataset.from_list(rows)\n\n    train_ds = to_ds(split[\"train\"])\n    val_ds = to_ds(split[\"val\"])\n    print(f\"[+] Split (seed={seed}, test held out in {test_inputs_path}):\")\n    print(f\"    Train : {len(train_ds)} examples\")\n    print(f\"    Val   : {len(val_ds)} examples\")\n    print(f\"    Test  : {len(split['test'])} examples (held out, not trained on)\")\n    return train_ds, val_ds, split\n\n\n# ── Precision / hardware autodetect (fix D1) ──────────────────────────────────\n\ndef detect_bf16() -> bool:\n    \"\"\"True on GPUs that support bfloat16 (e.g. A100); False on T4 (fp16 only).\"\"\"\n    import torch\n    return torch.cuda.is_available() and torch.cuda.is_bf16_supported()\n\n\n# ── Model + tokenizer loading (true QLoRA, fix D) ─────────────────────────────\n\ndef load_model_and_tokenizer(model_name: str, trust_remote_code: bool):\n    \"\"\"\n    Load Mistral-7B in 4-bit quantized form (true QLoRA setup).\n\n    BitsAndBytesConfig:\n      - load_in_4bit              : compress base weights to 4-bit (~4-5GB VRAM)\n      - bnb_4bit_quant_type=\"nf4\" : NormalFloat4, best quality for 4-bit\n      - bnb_4bit_use_double_quant : quantize the quantization constants too\n      - bnb_4bit_compute_dtype    : bf16 on A100, fp16 on T4 — MATCHED to the\n                                    mixed-precision used by the trainer (fix D1).\n    We do NOT set bnb_4bit_quant_storage (default uint8) and never use fp32\n    compute (fixes D2). After loading we run prepare_model_for_kbit_training so\n    LoRA trains correctly on the quantized base (fix D4).\n    \"\"\"\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n    from peft import prepare_model_for_kbit_training\n\n    bf16 = detect_bf16()\n    compute_dtype = torch.bfloat16 if bf16 else torch.float16\n    print(f\"[+] Precision: {'bf16' if bf16 else 'fp16'} \"\n          f\"(compute_dtype={compute_dtype})\")\n\n    print(f\"[+] Loading base model: {model_name}\")\n    print(\"    This downloads ~14GB on first run. Subsequent runs use cache.\")\n\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=compute_dtype,\n    )\n\n    model = AutoModelForCausalLM.from_pretrained(\n        model_name,\n        quantization_config=bnb_config,\n        device_map=\"auto\",                 # places layers on GPU automatically\n        trust_remote_code=trust_remote_code,\n    )\n\n    model.config.use_cache = False         # required for gradient checkpointing\n    model.config.pretraining_tp = 1\n\n    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=trust_remote_code)\n    # Mistral has no pad token by default; reuse EOS so batches can pad.\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = \"right\"       # right padding for causal-LM training\n\n    # k-bit prep: upcasts layernorms to fp32, enables input grads, and turns on\n    # gradient checkpointing — must run AFTER the 4-bit load and BEFORE LoRA.\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n\n    print(\"[+] Model loaded and prepared for k-bit training\")\n    return model, tokenizer, bf16\n\n\n# ── LoRA config (fix G2: target modules configurable) ─────────────────────────\n\ndef attach_lora(model, r, alpha, dropout, target_modules):\n    \"\"\"\n    Attach LoRA adapter matrices. Only A/B adapter weights train; the 4-bit base\n    is frozen. By default we adapt ALL linear projections (attention + MLP), which\n    generally fits a new output format better than attention-only.\n    \"\"\"\n    from peft import LoraConfig, get_peft_model, TaskType\n\n    lora_config = LoraConfig(\n        r=r,\n        lora_alpha=alpha,\n        target_modules=target_modules,\n        lora_dropout=dropout,\n        bias=\"none\",\n        task_type=TaskType.CAUSAL_LM,\n    )\n    model = get_peft_model(model, lora_config)\n\n    trainable, total = model.get_nb_trainable_parameters()\n    print(\"[+] LoRA adapters attached\")\n    print(f\"    Target modules   : {target_modules}\")\n    print(f\"    Trainable params : {trainable:,}  ({100 * trainable / total:.2f}% of total)\")\n    print(f\"    Total params     : {total:,}\")\n    return model\n\n\n# ── SFTConfig builder (branches on the installed TRL/transformers API) ─────────\n\ndef _sftconfig_fields():\n    from trl import SFTConfig\n    from dataclasses import fields as dc_fields\n    try:\n        return {f.name for f in dc_fields(SFTConfig)}\n    except TypeError:\n        # Not a dataclass on this version — fall back to signature inspection.\n        import inspect\n        return set(inspect.signature(SFTConfig.__init__).parameters)\n\n\ndef build_sft_config(output_dir, args, bf16, eval_mode=\"epoch\", max_steps=None):\n    \"\"\"\n    Build an SFTConfig, choosing field names that exist in the installed TRL:\n      * sequence length: `max_length` (TRL >=0.20) else `max_seq_length`\n      * eval cadence   : `eval_strategy` (newer) else `evaluation_strategy`\n      * `dataset_kwargs={'add_special_tokens': False}` when supported, since\n        build_chat already embeds BOS/EOS (avoids double special tokens).\n\n    eval_mode: \"epoch\" (full run), \"steps\" (smoke), or \"no\" (no eval).\n    \"\"\"\n    from trl import SFTConfig\n\n    valid = _sftconfig_fields()\n\n    kwargs = dict(\n        output_dir=output_dir,\n        per_device_train_batch_size=args.batch_size,\n        gradient_accumulation_steps=args.grad_accum,\n        optim=\"paged_adamw_8bit\",                       # fix G1\n        learning_rate=args.lr,\n        lr_scheduler_type=\"cosine\",\n        warmup_steps=5,\n        bf16=bf16,                                      # fix D1 (matched precision)\n        fp16=not bf16,\n        logging_steps=2 if max_steps else 10,\n        save_total_limit=2,\n        report_to=\"none\",\n        dataset_text_field=\"text\",\n        packing=False,\n        max_grad_norm=0.3,\n        gradient_checkpointing=True,                    # fix D4\n        seed=args.seed,                                 # fix G3\n    )\n    if \"data_seed\" in valid:\n        kwargs[\"data_seed\"] = args.seed                 # fix G3\n    if max_steps is not None:\n        kwargs[\"max_steps\"] = max_steps\n    else:\n        kwargs[\"num_train_epochs\"] = args.epochs\n\n    # Sequence length field name branch (fix C).\n    if \"max_length\" in valid:\n        kwargs[\"max_length\"] = args.max_seq_len\n    elif \"max_seq_length\" in valid:\n        kwargs[\"max_seq_length\"] = args.max_seq_len\n    else:\n        print(\"[!] SFTConfig exposes neither max_length nor max_seq_length; \"\n              \"examples may be truncated at the library default.\")\n\n    # Avoid double BOS/EOS: build_chat already adds them, so tell the SFT\n    # tokenization not to re-add special tokens (when the option exists).\n    if \"dataset_kwargs\" in valid:\n        kwargs[\"dataset_kwargs\"] = {\"add_special_tokens\": False}\n\n    if \"gradient_checkpointing_kwargs\" in valid:\n        kwargs[\"gradient_checkpointing_kwargs\"] = {\"use_reentrant\": False}\n\n    # Eval / checkpoint / best-model selection (fix F).\n    eval_key = \"eval_strategy\" if \"eval_strategy\" in valid else \"evaluation_strategy\"\n    if eval_mode == \"no\":\n        kwargs[eval_key] = \"no\"\n        kwargs[\"save_strategy\"] = \"no\"\n    elif eval_mode == \"steps\":\n        kwargs[eval_key] = \"steps\"\n        kwargs[\"eval_steps\"] = 2\n        kwargs[\"save_strategy\"] = \"steps\"\n        kwargs[\"save_steps\"] = 2\n    else:  # \"epoch\"\n        kwargs[eval_key] = \"epoch\"\n        kwargs[\"save_strategy\"] = \"epoch\"\n\n    if eval_mode != \"no\":\n        kwargs[\"per_device_eval_batch_size\"] = 1\n        kwargs[\"load_best_model_at_end\"] = True\n        kwargs[\"metric_for_best_model\"] = \"eval_loss\"\n        kwargs[\"greater_is_better\"] = False\n\n    # Drop anything the installed version doesn't accept, but warn so nothing is\n    # silently lost.\n    dropped = [k for k in kwargs if k not in valid]\n    for k in dropped:\n        kwargs.pop(k)\n    if dropped:\n        print(f\"[i] SFTConfig: ignored unsupported args on this version: {dropped}\")\n\n    return SFTConfig(**kwargs)\n\n\n# ── Completion-only collator (fix E1: in-context response-template ids) ────────\n\ndef build_collator(tokenizer, sample_text):\n    \"\"\"\n    Build a DataCollatorForCompletionOnlyLM so loss is computed only on the\n    completion (everything after [/INST]), not on the prompt.\n\n    CRITICAL: we pass the [/INST] token ids computed IN CONTEXT (see\n    prompt_format.response_template_ids), not the raw string, because \"[/INST]\"\n    can tokenize differently mid-sequence and otherwise the collator silently\n    masks the whole example.\n\n    Returns None (with a warning) if the collator isn't importable, so training\n    still runs (as plain LM loss) rather than crashing.\n    \"\"\"\n    try:\n        try:\n            from trl import DataCollatorForCompletionOnlyLM\n        except ImportError:\n            from trl.trainer import DataCollatorForCompletionOnlyLM\n    except Exception as e:\n        print(f\"[!] DataCollatorForCompletionOnlyLM unavailable ({e}); \"\n              \"falling back to plain LM loss (prompt NOT masked).\")\n        return None\n\n    response_ids = prompt_format.response_template_ids(tokenizer, sample_text)\n    print(f\"[+] Completion-only masking on response-template ids {response_ids} \"\n          f\"(in-context for '{prompt_format.RESPONSE_MARKER}')\")\n    return DataCollatorForCompletionOnlyLM(response_template=response_ids, tokenizer=tokenizer)\n\n\ndef check_label_masking(tokenizer, collator, text):\n    \"\"\"\n    Verify the collator masks the prompt (-100) and keeps real ids on the\n    completion (fix E acceptance). Returns (ok: bool, detail: str).\n    \"\"\"\n    enc = tokenizer(text, add_special_tokens=False)\n    batch = collator([{\"input_ids\": enc[\"input_ids\"]}])\n    labels = batch[\"labels\"][0].tolist()\n\n    real = [i for i, l in enumerate(labels) if l != -100]\n    if not real:\n        return False, \"ALL tokens masked — collator failed to find the response template.\"\n    first_real = real[0]\n    prefix_all_masked = all(l == -100 for l in labels[:first_real])\n    n_real = len(real)\n    detail = (f\"first unmasked token at index {first_real}/{len(labels)}; \"\n              f\"{n_real} completion tokens carry loss; prompt fully masked={prefix_all_masked}\")\n    return (prefix_all_masked and n_real > 0), detail\n\n\n# ── Trainer build (branches on processing_class vs tokenizer) ──────────────────\n\ndef build_trainer(model, tokenizer, sft_config, train_ds, eval_ds, collator):\n    from trl import SFTTrainer\n    from transformers import EarlyStoppingCallback\n\n    callbacks = []\n    if eval_ds is not None:\n        callbacks.append(EarlyStoppingCallback(early_stopping_patience=2))  # fix F2\n\n    common = dict(model=model, args=sft_config, train_dataset=train_ds)\n    if eval_ds is not None:\n        common[\"eval_dataset\"] = eval_ds\n    if collator is not None:\n        common[\"data_collator\"] = collator\n    if callbacks:\n        common[\"callbacks\"] = callbacks\n\n    # TRL renamed the tokenizer argument across versions: newer uses\n    # processing_class, older uses tokenizer. Prefer the new name.\n    try:\n        return SFTTrainer(processing_class=tokenizer, **common)\n    except TypeError:\n        return SFTTrainer(tokenizer=tokenizer, **common)\n\n\n# ── Post-training self-test (fix P3) ──────────────────────────────────────────\n\ndef post_training_selftest(model, tokenizer, test_rec, max_new_tokens):\n    \"\"\"\n    Generate one held-out sample and assert (i) all three headers present in\n    order and (ii) EOS reached before max_new_tokens. Prints PASS/FAIL.\n    \"\"\"\n    import torch\n\n    model.config.use_cache = True\n    if hasattr(model, \"gradient_checkpointing_disable\"):\n        model.gradient_checkpointing_disable()\n    model.eval()\n\n    prompt = prompt_format.build_chat(test_rec, tokenizer, for_training=False)\n    inputs = tokenizer(prompt, return_tensors=\"pt\", add_special_tokens=False).to(model.device)\n    n = inputs[\"input_ids\"].shape[1]\n    with torch.no_grad():\n        out = model.generate(\n            **inputs,\n            max_new_tokens=max_new_tokens,\n            do_sample=False,\n            eos_token_id=tokenizer.eos_token_id,\n            pad_token_id=tokenizer.pad_token_id,\n        )\n    new_ids = out[0][n:].tolist()\n    text = tokenizer.decode(new_ids, skip_special_tokens=True)\n\n    positions = [text.find(h) for h in REQUIRED_HEADERS]\n    headers_ok = all(p >= 0 for p in positions) and positions == sorted(positions)\n    eos_hit = tokenizer.eos_token_id in new_ids\n\n    print()\n    print(\"─\" * 60)\n    print(\"[self-test] held-out generation\")\n    print(f\"    headers present & in order : {headers_ok}  ({positions})\")\n    print(f\"    EOS reached before cap     : {eos_hit}  \"\n          f\"(generated {len(new_ids)}/{max_new_tokens} tokens)\")\n    ok = headers_ok and eos_hit\n    print(f\"    RESULT: {'PASS' if ok else 'FAIL'}\")\n    print(\"─\" * 60)\n    return ok\n\n\n# ── Optional adapter merge (fix P4) ───────────────────────────────────────────\n\ndef merge_adapter(base_model, adapter_dir, out_dir, trust_remote_code):\n    \"\"\"\n    Merge the LoRA adapter into a fresh fp16 copy of the base model and save it\n    alongside the adapter. NOTE: this loads the FULL (non-quantized) base in\n    fp16, so it needs ~14-16GB of RAM/VRAM — far more than 4-bit training.\n    \"\"\"\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from peft import PeftModel\n\n    print(\"[+] --merge: loading base in fp16 to merge the adapter \"\n          \"(needs ~14-16GB; off by default)…\")\n    base = AutoModelForCausalLM.from_pretrained(\n        base_model, torch_dtype=torch.float16, device_map=\"auto\",\n        trust_remote_code=trust_remote_code,\n    )\n    merged = PeftModel.from_pretrained(base, adapter_dir)\n    merged = merged.merge_and_unload()\n    mdir = Path(out_dir) / \"merged\"\n    merged.save_pretrained(mdir, safe_serialization=True)\n    AutoTokenizer.from_pretrained(base_model, trust_remote_code=trust_remote_code).save_pretrained(mdir)\n    print(f\"[✓] Merged fp16 model saved to: {mdir}\")\n\n\n# ── Training orchestration ────────────────────────────────────────────────────\n\ndef train(args):\n    \"\"\"Run the supervised fine-tuning loop using SFTTrainer (completion-only).\"\"\"\n    check_imports()\n    from transformers import set_seed\n    set_seed(args.seed)                                # fix G3\n\n    model, tokenizer, bf16 = load_model_and_tokenizer(args.base, args.trust_remote_code)\n    model = attach_lora(model, args.r, args.alpha, args.lora_dropout, args.target_modules)\n\n    train_ds, val_ds, split = load_dataset_split(\n        args.data, tokenizer, args.test_inputs,\n        args.test_size, args.val_frac, args.seed,\n    )\n    eval_enabled = len(val_ds) >= 1\n    if not eval_enabled:\n        print(\"[!] Validation split is empty — eval/early-stopping disabled.\")\n    if len(train_ds) + len(val_ds) < 100:\n        print(\"[i] Note: this is a small dataset; eval_loss will be noisy.\")\n\n    # True optimizer-step count (fix J3).\n    eff_batch = args.batch_size * args.grad_accum\n    steps_per_epoch = math.ceil(len(train_ds) / eff_batch)\n    total_steps = steps_per_epoch * args.epochs\n    print()\n    print(\"[+] Starting training…\")\n    print(f\"    Output dir       : {args.out}\")\n    print(f\"    Epochs           : {args.epochs}\")\n    print(f\"    Effective batch  : {eff_batch} (bs {args.batch_size} x accum {args.grad_accum})\")\n    print(f\"    Optimizer steps  : ~{total_steps} ({steps_per_epoch}/epoch x {args.epochs})\")\n    print()\n\n    sft_config = build_sft_config(args.out, args, bf16,\n                                  eval_mode=\"epoch\" if eval_enabled else \"no\")\n\n    sample_text = train_ds[0][\"text\"]\n    collator = build_collator(tokenizer, sample_text)\n    if collator is not None:\n        ok, detail = check_label_masking(tokenizer, collator, sample_text)\n        print(f\"[{'+' if ok else '!'}] label-masking check: {detail}\")\n        if not ok:\n            print(\"[!] Completion-only masking looks wrong — aborting before wasting a GPU run.\")\n            sys.exit(1)\n\n    trainer = build_trainer(model, tokenizer, sft_config, train_ds,\n                            val_ds if eval_enabled else None, collator)\n\n    trainer.train()\n\n    if eval_enabled:\n        best = getattr(trainer.state, \"best_model_checkpoint\", None)\n        best_metric = getattr(trainer.state, \"best_metric\", None)\n        print(f\"[+] Best checkpoint: {best}  (eval_loss={best_metric})\")\n\n    print(\"[+] Saving final adapter weights…\")\n    trainer.save_model(args.out)\n    tokenizer.save_pretrained(args.out)\n    print(f\"[✓] Model saved to: {args.out}\")\n\n    ok = True\n    if split[\"test\"]:\n        ok = post_training_selftest(model, tokenizer, split[\"test\"][0], args.max_new_tokens)\n\n    if args.merge:\n        merge_adapter(args.base, args.out, args.out, args.trust_remote_code)\n\n    print()\n    print(\"Next step: run evaluation/evaluate.py to score the fine-tuned model\")\n    if not ok:\n        sys.exit(1)\n\n\n# ── --check_data probe (fix P1; deps-free) ────────────────────────────────────\n\ndef run_check_data(args) -> int:\n    print(\"=\" * 60)\n    print(\"  --check_data : data + prompt-parity probe (no model load)\")\n    print(\"=\" * 60)\n\n    if not Path(args.data).exists():\n        print(f\"[!] Dataset not found: {args.data}\")\n        return 1\n\n    records = data_split.load_records(args.data)\n    print(f\"[+] Loaded {len(records)} records from {args.data}\")\n\n    # Try a real tokenizer; fall back to a labelled chars/4 estimate.\n    tokenizer = None\n    try:\n        from transformers import AutoTokenizer\n        tokenizer = AutoTokenizer.from_pretrained(\n            args.base, trust_remote_code=args.trust_remote_code)\n        print(f\"[+] Using REAL tokenizer: {args.base}\")\n    except Exception as e:\n        print(f\"[i] Real tokenizer unavailable ({type(e).__name__}); \"\n              \"using chars/4 ESTIMATE for token counts.\")\n\n    # 1) Train/inference parity (fix A acceptance).\n    prompt_format.assert_parity(records, tokenizer)\n    print(\"[+] PARITY OK: inference prompt is a strict prefix of training text \"\n          \"for all records (single shared build_chat).\")\n\n    # 2) Token-length stats + truncation flag (fix C acceptance).\n    if tokenizer is not None:\n        lengths = [len(tokenizer(prompt_format.build_chat(r, tokenizer, True),\n                                 add_special_tokens=False)[\"input_ids\"]) for r in records]\n        label = \"real tokenizer\"\n    else:\n        lengths = [estimate_tokens(prompt_format.build_chat(r, None, True)) for r in records]\n        label = \"chars/4 ESTIMATE\"\n    s = sorted(lengths)\n    p = lambda q: s[min(len(s) - 1, int(len(s) * q))]\n    print(f\"[+] Full training-text token length ({label}): \"\n          f\"min={s[0]} p50={p(0.5)} p90={p(0.9)} max={s[-1]}\")\n    over = sum(1 for L in lengths if L > args.max_seq_len)\n    print(f\"[+] Configured max_length = {args.max_seq_len}; \"\n          f\"examples that would be truncated: {over}/{len(lengths)}\")\n\n    # 3) Completion-only masking check (needs trl + real tokenizer).\n    if tokenizer is not None:\n        sample_text = prompt_format.build_chat(records[0], tokenizer, True)\n        collator = build_collator(tokenizer, sample_text)\n        if collator is not None:\n            all_ok = True\n            for r in records[:2]:\n                t = prompt_format.build_chat(r, tokenizer, True)\n                ok, detail = check_label_masking(tokenizer, collator, t)\n                print(f\"    [{'+' if ok else '!'}] masking: {detail}\")\n                all_ok = all_ok and ok\n            if not all_ok:\n                print(\"[!] label-masking check FAILED.\")\n                return 1\n        else:\n            print(\"[i] masking check skipped (collator unavailable).\")\n    else:\n        print(\"[i] masking check skipped (no transformers/trl installed).\")\n\n    # 4) Split / unseen-test intersection (fix H acceptance).\n    split = data_split.ensure_test_inputs(\n        args.data, args.test_inputs, args.test_size, args.val_frac, args.seed)\n    train_ids = data_split.ids_of(split[\"train\"]) | data_split.ids_of(split[\"val\"])\n    test_ids = data_split.ids_of(split[\"test\"])\n    disjoint = train_ids.isdisjoint(test_ids)\n    print(f\"[+] Split: train {len(split['train'])} / val {len(split['val'])} / \"\n          f\"test {len(split['test'])} (artifact: {args.test_inputs})\")\n    print(f\"[{'+' if disjoint else '!'}] train ∩ test ids empty: {disjoint}\")\n    if not disjoint:\n        return 1\n\n    # Final gate: truncation must be zero at the configured length.\n    if over > 0:\n        print(f\"[!] {over} example(s) exceed max_length={args.max_seq_len}. \"\n              \"Raise --max_seq_len or shorten the data before training.\")\n        return 1\n\n    print(\"\\n[✓] check_data: ALL CHECKS PASSED\")\n    return 0\n\n\n# ── --smoke probe (fix P2; CUDA-guarded) ──────────────────────────────────────\n\ndef run_smoke(args) -> int:\n    print(\"=\" * 60)\n    print(\"  --smoke : 3-step end-to-end training + 1 generation\")\n    print(\"=\" * 60)\n    try:\n        import torch\n    except ImportError:\n        print(\"[smoke] PyTorch is not installed here — smoke is a NO-OP.\")\n        print(\"        On Colab with a GPU this runs max_steps=3 training + one\")\n        print(\"        generation to exercise the whole path fast.\")\n        return 0\n    if not torch.cuda.is_available():\n        print(\"[smoke] No CUDA GPU detected — smoke is a NO-OP.\")\n        print(\"        On Colab with a GPU this runs max_steps=3 training + one\")\n        print(\"        generation to exercise the whole path fast.\")\n        return 0\n\n    check_imports()\n    from transformers import set_seed\n    set_seed(args.seed)\n\n    model, tokenizer, bf16 = load_model_and_tokenizer(args.base, args.trust_remote_code)\n    model = attach_lora(model, args.r, args.alpha, args.lora_dropout, args.target_modules)\n    train_ds, val_ds, split = load_dataset_split(\n        args.data, tokenizer, args.test_inputs,\n        args.test_size, args.val_frac, args.seed)\n\n    sft_config = build_sft_config(args.out, args, bf16, eval_mode=\"steps\", max_steps=3)\n    sample_text = train_ds[0][\"text\"]\n    collator = build_collator(tokenizer, sample_text)\n    if collator is not None:\n        ok, detail = check_label_masking(tokenizer, collator, sample_text)\n        print(f\"[{'+' if ok else '!'}] label-masking check: {detail}\")\n\n    trainer = build_trainer(model, tokenizer, sft_config, train_ds, val_ds, collator)\n    trainer.train()\n    best = getattr(trainer.state, \"best_model_checkpoint\", None)\n    best_metric = getattr(trainer.state, \"best_metric\", None)\n    print(f\"[smoke] best checkpoint: {best} (eval_loss={best_metric})\")\n\n    if split[\"test\"]:\n        post_training_selftest(model, tokenizer, split[\"test\"][0],\n                               max_new_tokens=min(args.max_new_tokens, 256))\n    print(\"[✓] smoke complete\")\n    return 0\n\n\n# ── target-module parsing (fix G2) ────────────────────────────────────────────\n\ndef parse_target_modules(value: str):\n    if value == \"all\":\n        return ALL_LINEAR\n    if value == \"attn\":\n        return ATTN_ONLY\n    return [m.strip() for m in value.split(\",\") if m.strip()]\n\n\n# ── Main ──────────────────────────────────────────────────────────────────────\n\ndef main():\n    parser = argparse.ArgumentParser(description=\"QLoRA fine-tuning for scenario generation\")\n    parser.add_argument(\"--data\", default=\"dataset/training_data.jsonl\",\n                        help=\"Path to training_data.jsonl\")\n    parser.add_argument(\"--out\", default=\"fine_tuning/output\",\n                        help=\"Output directory for adapter weights\")\n    parser.add_argument(\"--base\", default=BASE_MODEL, help=f\"Base model (default: {BASE_MODEL})\")\n\n    # Hyperparameters (fix G5) — current values as defaults.\n    parser.add_argument(\"--epochs\", type=int, default=3)\n    parser.add_argument(\"--lr\", type=float, default=2e-4)\n    parser.add_argument(\"--r\", type=int, default=16, help=\"LoRA rank\")\n    parser.add_argument(\"--alpha\", type=int, default=32, help=\"LoRA alpha\")\n    parser.add_argument(\"--lora_dropout\", type=float, default=0.05)\n    parser.add_argument(\"--max_seq_len\", type=int, default=MAX_SEQ_LEN)\n    parser.add_argument(\"--target_modules\", default=\"all\",\n                        help='\"all\" (default, all linear), \"attn\" (attention-only), '\n                             \"or a comma list e.g. q_proj,v_proj\")\n    parser.add_argument(\"--batch_size\", type=int, default=1)\n    parser.add_argument(\"--grad_accum\", type=int, default=4)\n\n    # Split / reproducibility.\n    parser.add_argument(\"--test_inputs\", default=DEFAULT_TEST_INPUTS,\n                        help=\"Held-out test artifact (shared with evaluation)\")\n    parser.add_argument(\"--test_size\", type=int, default=10, help=\"Held-out test count\")\n    parser.add_argument(\"--val_frac\", type=float, default=0.15,\n                        help=\"Validation fraction of the train pool\")\n    parser.add_argument(\"--seed\", type=int, default=42)\n\n    # Behaviour flags.\n    parser.add_argument(\"--trust_remote_code\", action=\"store_true\",\n                        help=\"Pass trust_remote_code=True (unnecessary for Mistral)\")\n    parser.add_argument(\"--max_new_tokens\", type=int, default=1280,\n                        help=\"Tokens to generate in the post-training self-test\")\n    parser.add_argument(\"--merge\", action=\"store_true\",\n                        help=\"Also save a merged fp16 model (needs ~14-16GB)\")\n    parser.add_argument(\"--check_data\", action=\"store_true\",\n                        help=\"Offline data + prompt-parity probe (no model load)\")\n    parser.add_argument(\"--smoke\", action=\"store_true\",\n                        help=\"3-step CUDA-guarded end-to-end run (no-op without a GPU)\")\n\n    args = parser.parse_args()\n    args.target_modules = parse_target_modules(args.target_modules)\n\n    # Offline probes first (these never load the 7B model unless a GPU is present).\n    if args.check_data:\n        sys.exit(run_check_data(args))\n    if args.smoke:\n        sys.exit(run_smoke(args))\n\n    # ── Real training path ──\n    check_imports()                      # friendly message if the ML stack is absent\n    import torch\n    if not torch.cuda.is_available():\n        print(\"[!] WARNING: No GPU detected.\")\n        print(\"    QLoRA fine-tuning needs a GPU with >=16GB VRAM (Colab T4 / Kaggle).\")\n        print(\"    Continuing anyway — will be extremely slow on CPU.\")\n        print()\n\n    print(\"=\" * 60)\n    print(\"  QLoRA Fine-Tuning — Cyber Attack Scenario Generation\")\n    print(\"=\" * 60)\n    print(f\"  Base model : {args.base}\")\n    print(f\"  Dataset    : {args.data}\")\n    print(f\"  Output     : {args.out}\")\n    print(\"=\" * 60)\n    print()\n\n    if not Path(args.data).exists():\n        print(f\"[!] Dataset not found: {args.data}\")\n        print(\"    Run generate_outputs.py first.\")\n        sys.exit(1)\n\n    Path(args.out).mkdir(parents=True, exist_ok=True)\n    train(args)\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
pathlib.Path("fine_tune.py").write_text("\"\"\"Compatibility launcher for the QLoRA fine-tuning script.\n\nThis file lets Colab and local commands run:\n    python fine_tune.py --data training_data.jsonl --out ./output\n\"\"\"\n\nfrom finetuning import main\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
pathlib.Path("requirements_finetune.txt").write_text("# Fine-tuning dependencies\n# Install with: pip install -r fine_tuning/requirements_finetune.txt\n#\n# Pinned to the stack that produced the shipped adapter (see output/README.md):\n#   Transformers 5.x, TRL 0.29.x, PEFT 0.18.x, datasets 4.x, torch 2.10, bnb 0.49.x.\n# The code branches on the TRL/transformers API differences (max_length vs\n# max_seq_length, processing_class vs tokenizer, the completion-only collator),\n# so nearby versions also work; these ranges are the verified-compatible set.\n\ntorch>=2.4\ntransformers>=5.0,<6\ntrl>=0.29,<0.30\npeft>=0.18,<0.19\ndatasets>=4.0,<5\naccelerate>=1.1\nbitsandbytes>=0.47,<0.50\n", encoding='utf-8')
print('Wrote:', "prompt_format.py", "data_split.py", "finetuning.py", "fine_tune.py", "requirements_finetune.txt")

In [ ]:
from google.colab import files
print('Upload training_data.jsonl')
up = files.upload()
import os; print('cwd files:', os.listdir('.'))

In [ ]:
# Runs the fixed QLoRA pipeline. finetuning.py auto-creates the held-out
# split at --test_inputs and trains on the complement only.
!python fine_tune.py --data training_data.jsonl --out ./output --test_inputs ./test_inputs.jsonl
# Override examples:
#   --target_modules attn   (attention-only LoRA)
#   --epochs 5 --lr 1e-4 --r 32 --alpha 64
#   --merge                 (also save a merged fp16 model; needs ~16GB)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('finetuned_adapters', 'zip', './output')
files.download('finetuned_adapters.zip')
# Keep the SAME split for evaluation:
files.download('test_inputs.jsonl')